# Probleme

- le probleme annotation 
- starsolo prend fichier annotation et genome puis ressort un feature tsv : 
gene-QLH64_00005	QLH64_00005	Gene Expression
gene-QLH64_00010	QLH64_00010	Gene Expression
gene-QLH64_00015	QLH64_00015	Gene Expression
gene-QLH64_00020	QLH64_00020	Gene Expression
gene-QLH64_00025	rrf	Gene Expression
gene-QLH64_00030	recC	Gene Expression
gene-QLH64_00035	recB	Gene Expression
gene-QLH64_00040	recD	Gene Expression
gene-QLH64_00045	QLH64_00045	Gene Expression


le probleme est que la deuxieme je ne veux pas les symboles mais les noms des genes




# Import R packages 

In [1]:
# Load libraries
library(Seurat) # For single-cell RNA-seq data manipulation
library(Matrix) # For matrix manipulation


Loading required package: SeuratObject

Loading required package: sp


Attaching package: 'SeuratObject'


The following objects are masked from 'package:base':

    intersect, t




Error in .make_numeric_version(x, strict, .standard_regexps()$valid_numeric_version) : 
  invalid non-character version specification 'x' (type: double)


## Set the working directory

In [2]:
setwd("/workspaces/microsplit_DOL")

In [3]:
# Define the data directory
data_dir <- "data/big_data/raw_starsolo"

# Verify if the data directory exists
if (!dir.exists(data_dir)) {
    stop("The specified directory does not exist: ", data_dir)
}


In [4]:
# output_file <- "data/processed/Seurat_object/SeuratObject_All.rds"
output_file <- "data/big_data/RDS_files/SeuratObject_All.rds"


## Import neccesary files

In [5]:
# _____Attention_____
# float error STARsolo UniqueAndMult-Uniform.mtx which is a float matrix that cause error in Seurat ReadMtx, ReadSARsolo, Read10X...

# https://github.com/alexdobin/STAR/issues/1397




# Construct the file paths
matrix_file <- file.path(data_dir, "UniqueAndMult-Uniform.mtx")
# awk '{if (NR>3) $3=int($3+0.5); print}' matrix.mtx | sed 's/float/integer/' > matrix_rounded.mtx
# change float to integer without rounding
output_matrix <- sub(".mtx$", "_integer.mtx", matrix_file) #
system(paste("sed '1,3s/float/integer/'", matrix_file, ">", output_matrix))

matrix_file <- output_matrix

features_file <- file.path(data_dir, "features.tsv")
barcodes_file <- file.path(data_dir, "barcodes.tsv")

# Verify if the necessary files exist
if (!file.exists(matrix_file)) stop("Missing file: ", matrix_file)
if (!file.exists(features_file)) stop("Missing file: ", features_file)
if (!file.exists(barcodes_file)) stop("Missing file: ", barcodes_file)

In [6]:
str(features_file)

features <- read.delim(features_file, header = FALSE)

str(features)


 chr "data/big_data/raw_starsolo/features.tsv"
'data.frame':	6248 obs. of  3 variables:
 $ V1: chr  "gene-QLH64_00005" "gene-QLH64_00010" "gene-QLH64_00015" "gene-QLH64_00020" ...
 $ V2: chr  "QLH64_00005" "QLH64_00010" "QLH64_00015" "QLH64_00020" ...
 $ V3: chr  "Gene Expression" "Gene Expression" "Gene Expression" "Gene Expression" ...


In [7]:
# Modifier la 1ère colonne pour remplacer les underscores par des tirets
features$V1 <- gsub("_", "-", features$V1)

# Modifier la 2e colonne à partir de la 1ère
features$V2 <- gsub("_", "-", sub("gene-", "", features$V1))

# (Optionnel) Afficher pour vérifier
print(features)

# save the modified features to a new file
modified_features_file <- sub(".tsv$", "_modified.tsv", features_file)
write.table(features, file = modified_features_file, sep = "\t", row.names = FALSE, col.names = FALSE, quote = FALSE)


                   V1          V2              V3
1    gene-QLH64-00005 QLH64-00005 Gene Expression
2    gene-QLH64-00010 QLH64-00010 Gene Expression
3    gene-QLH64-00015 QLH64-00015 Gene Expression
4    gene-QLH64-00020 QLH64-00020 Gene Expression
5    gene-QLH64-00025 QLH64-00025 Gene Expression
6    gene-QLH64-00030 QLH64-00030 Gene Expression
7    gene-QLH64-00035 QLH64-00035 Gene Expression
8    gene-QLH64-00040 QLH64-00040 Gene Expression
9    gene-QLH64-00045 QLH64-00045 Gene Expression
10   gene-QLH64-00050 QLH64-00050 Gene Expression
11   gene-QLH64-00055 QLH64-00055 Gene Expression
12   gene-QLH64-00060 QLH64-00060 Gene Expression
13   gene-QLH64-00065 QLH64-00065 Gene Expression
14   gene-QLH64-00070 QLH64-00070 Gene Expression
15   gene-QLH64-00075 QLH64-00075 Gene Expression
16   gene-QLH64-00080 QLH64-00080 Gene Expression
17   gene-QLH64-00085 QLH64-00085 Gene Expression
18   gene-QLH64-00090 QLH64-00090 Gene Expression
19   gene-QLH64-00095 QLH64-00095 Gene Expression


# Create the Seurat object with barcodes columns

In [8]:


# Start timing the main process
cat("\nStarting data processing...\n")
processing_time <- system.time({
    # Read the data using Seurat's ReadMtx function
    cat("Reading data with ReadMtx...\n")
    expression_matrix <- ReadMtx(
        mtx = matrix_file,
        features = modified_features_file,
        cells = barcodes_file
    )

    # Create Seurat object
    cat("Creating Seurat object...\n")
    seurat_object <- CreateSeuratObject(counts = expression_matrix)

    # Remove orig.ident column if it exists
    if ("orig.ident" %in% colnames(seurat_object@meta.data)) {
        seurat_object$orig.ident <- NULL
    }

    # Add barcode columns by extracting parts from the barcodes
    cat("Extracting barcode information...\n")
    barcodes <- colnames(expression_matrix)
    barcode_parts <- strsplit(barcodes, "_")

    # Extract rd1, rd2, and rd3 barcodes
    barcode_rd1 <- sapply(barcode_parts, function(x) x[length(x)])
    barcode_rd2 <- sapply(barcode_parts, function(x) x[2])
    barcode_rd3 <- sapply(barcode_parts, function(x) x[1])

    # Add barcode columns to Seurat object
    seurat_object$barcode.rd1 <- barcode_rd1
    seurat_object$barcode.rd2 <- barcode_rd2
    seurat_object$barcode.rd3 <- barcode_rd3

    # Print diagnostic information about barcode extraction
    cat("\nBarcode extraction information:\n")
    cat("Number of unique rd1 barcodes:", length(unique(barcode_rd1)), "\n")
    cat("Number of unique rd2 barcodes:", length(unique(barcode_rd2)), "\n")
    cat("Number of unique rd3 barcodes:", length(unique(barcode_rd3)), "\n")
    cat("\nFirst few barcodes:\n")
    print(head(data.frame(
        rd1 = barcode_rd1,
        rd2 = barcode_rd2,
        rd3 = barcode_rd3
    )))
})



Starting data processing...
Reading data with ReadMtx...
Creating Seurat object...
Extracting barcode information...

Barcode extraction information:
Number of unique rd1 barcodes: 90 
Number of unique rd2 barcodes: 96 
Number of unique rd3 barcodes: 95 

First few barcodes:
       rd1      rd2      rd3
1 AAACGATA AAACATCG AAACATCG
2 AAACGATA AAACATCG AACAACCA
3 AAACGATA AAACATCG AACCGAGA
4 AAACGATA AAACATCG AACGCTTA
5 AAACGATA AAACATCG AACGTGAT
6 AAACGATA AAACATCG AACTCACC


In [9]:

# Print timing information
cat("\nProcessing time:\n")
cat("User time:", processing_time["user.self"], "seconds\n")
cat("System time:", processing_time["sys.self"], "seconds\n")
cat("Elapsed time:", processing_time["elapsed"], "seconds\n")

# Print a summary of the Seurat object
print(seurat_object)

# create directory if not exists
dir.create("data/big_data/RDS_files", showWarnings = FALSE, recursive = TRUE)
# Save the Seurat object to an RDS file

saveRDS(seurat_object, file = output_file)

# Verify if the file was successfully created
if (file.exists(output_file)) {
    # Get file information
    file_info <- file.info(output_file)

    # Display file information
    cat("File successfully created:", output_file, "\n")
    cat("File size:", round(file_info$size / 1024 / 1024, 2), "MB\n")
    cat("Creation date:", format(file_info$mtime, "%Y-%m-%d %H:%M:%S"), "\n")

    # Display Seurat object dimensions
    cat("\nSeurat object information:\n")
    cat("Number of cells:", ncol(seurat_object), "\n")
    cat("Number of genes:", nrow(seurat_object), "\n")
} else {
    cat("Error: The file was not created.")
}



Processing time:
User time: 65.042 seconds
System time: 20.05 seconds
Elapsed time: 86.337 seconds
An object of class Seurat 
6248 features across 820800 samples within 1 assay 
Active assay: RNA (6248 features, 0 variable features)
 1 layer present: counts
File successfully created: data/big_data/RDS_files/SeuratObject_All.rds 
File size: 73.7 MB
Creation date: 2025-06-13 12:48:17 

Seurat object information:
Number of cells: 820800 
Number of genes: 6248 


In [10]:
# Generate a summary of the Seurat object
seurat_object
head(seurat_object@meta.data)


An object of class Seurat 
6248 features across 820800 samples within 1 assay 
Active assay: RNA (6248 features, 0 variable features)
 1 layer present: counts

,nCount_RNA,nFeature_RNA,barcode.rd1,barcode.rd2,barcode.rd3
,<dbl>,<int>,<chr>,<chr>,<chr>
AAACATCG_AAACATCG_AAACGATA,49,44,AAACGATA,AAACATCG,AAACATCG
AACAACCA_AAACATCG_AAACGATA,8,14,AAACGATA,AAACATCG,AACAACCA
AACCGAGA_AAACATCG_AAACGATA,33,34,AAACGATA,AAACATCG,AACCGAGA
AACGCTTA_AAACATCG_AAACGATA,119,89,AAACGATA,AAACATCG,AACGCTTA
AACGTGAT_AAACATCG_AAACGATA,2,10,AAACGATA,AAACATCG,AACGTGAT
AACTCACC_AAACATCG_AAACGATA,1,5,AAACGATA,AAACATCG,AACTCACC


In [11]:
# print genenames
cat("\nFirst few gene names:\n")
head(rownames(seurat_object))



First few gene names:


[1] "QLH64-00005" "QLH64-00010" "QLH64-00015" "QLH64-00020" "QLH64-00025"
[6] "QLH64-00030"